In [1]:
import math
import torch

In large neural networks, the number of parameters becomes very large. Performing gradient descent will hence require many derivatives. An efficient way to find the derivatives is to do a forward pass when calculating the loss function to track how the parameters are related and then a backward pass where we calculate the derivative with respect to each parameter starting from the loss and working backwards, using the chain-rule to calculate the derivatives. This can be visualised by a graph. I implemented the micrograd here to help myself understand how autograd in PyTorch works.

In [2]:
class Value:
    def __init__(self,data,parent=()):
        self.data=data
        self.parent=parent
        self.grad=0
        self._backward=lambda: None

The value class is the node in the graph. There will be a node for each parameter in our model; each of these stores the actual value of the parameter, as well as the parent nodes (not necessarily a parameter). The grad is initialised to 0 at the start but will accumulate when we back propagate. The ._backward will find the derivative with respect to the parents and update the parents' grads, if the node is a leaf node this will do nothing.

In the code below, we check if the input is an instance of Value and change it to be one if not so that we can add non Value instances to a Value instance, for convenience.

The _backward method will update the parents' grad value; this is an application of the chain rule.

Addition example:

$x=a+b$, $\frac{\partial x}{\partial a}=1$, so if Loss=$f(x)$, 
$\frac{\partial f}{\partial a}=\frac{\partial f}{\partial x}\frac{\partial x}{\partial a}=\frac{\partial f}{\partial x}$, b works analagously. 

The reason for += is because there could be multiple nodes with a or b as a parent. Each of these nodes will also contribute to the final partial derivate with respect to a hence we accumulate these.

In general $\frac{\partial f}{\partial a}=\sum_L \frac{\partial f}{\partial L}\frac{\partial L}{\partial a}$ for all L such that a is a parent of L.

In [3]:
def add(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    out=Value(self.data+other.data,(self,other))
    def _backward():
        self.grad+=out.grad
        other.grad+=out.grad
    out._backward=_backward
    return out

Value.__add__=add

In [4]:
def mul(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    out=Value(self.data*other.data,(self,other))
    def _backward():
        self.grad+=other.data*out.grad
        other.grad+=self.data*out.grad
    out._backward=_backward
    return out

Value.__mul__=mul

In [5]:
def power(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    out=Value(self.data**other.data,(self,other))
    def _backward():
        self.grad+=other.data*(self.data**(other.data-1))*out.grad
        other.grad+=math.log(self.data)*(out.data)*out.grad
    out._backward=_backward
    return out

Value.__pow__=power

In [6]:
def truediv(self,other):
    return self*(other**-1)

Value.__truediv__=truediv

In [7]:
def negate(self):
    return self*(-1)
    
def subtract(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    return self+-other

Value.__neg__=negate
Value.__sub__=subtract

In [8]:
def radd(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    return other+self
    
def rsubtract(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    return -other+self

def rmul(self,other):
    if not isinstance(other,Value):
        other=Value(other)
    return other*self

Value.__radd__=radd
Value.__rsub__=rsubtract
Value.__rmul__=rmul

The above are just reverse versions of addition, subtraction and multiplication respectively. We try these when the original versions don't work (the first input is not a Value instance).

In [9]:
def exp(self):
    out=Value(math.exp(self.data),(self,))
    def _backward():
        self.grad+=out.data*out.grad
    out._backward=_backward
    return out

Value.exp=exp

Below is the backward method. We first do a topological sort so that when we apply the _backward() method to the nodes, the node will definitely have its grad calculated correctly so the parent nodes can also get updated correctly.

In [10]:
def backward(self):
    self.grad=1
    topo=[]
    visited=set()
    def topo_build(v):
        if v not in visited:
            visited.add(v)
            for parent in v.parent:
                topo_build(parent)
            topo.append(v)
    topo_build(self)
    for value in reversed(topo):
        value._backward()

Value.backward=backward

In [11]:
def zero_grad(self):
    self.grad=0

Value.zero_grad=zero_grad

In [12]:
a=Value(5)
b=Value(3)
c=a*b+4
d=c**2
print(d.data)
d.backward()
print(a.grad,b.grad,c.grad)

f=torch.tensor([5],dtype=torch.float32,requires_grad=True)
g=torch.tensor([3],dtype=torch.float32,requires_grad=True)
h=f*g+4
i=h**2
print(i.item())
i.backward()
print(f.grad,g.grad,h.grad)

361
114 190 38
361.0
tensor([114.]) tensor([190.]) None


/var/folders/_g/v6t1k1_d41zdmlpt33m40vdc0000gn/T/ipykernel_87167/1338629757.py:15: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:497.)
  print(f.grad,g.grad,h.grad)


We see for the leaf nodes, my micrograd and PyTorch implementation have the same grad. PyTorch doesn't store the grad with respect to h as its not a leaf node so whilst the gradient is calculated as an intermediate step, it is discarded to save memory.